# 🔁 Laboratório de CDC (`ingest/generate_cdc_updates.py`) — upsert sem mudar o pipeline

**O que este notebook é:** a prova, em memória, da afirmação mais forte do Passo 9 — *o pipeline não precisa mudar para absorver CDC*. Um update (mesmo `event_id`, conteúdo novo, timestamp maior) é engolido pela mesma dedup determinística do contrato, por construção.

**O que ele não é:** DMS/Debezium de verdade (o LocalStack community não emula DMS — o desenho completo está em `docs/CDC-DMS.md`). Aqui provamos a **semântica**: upsert por chave + ordenação temporal. A ferramenta de captura é intercambiável; a semântica é o contrato.

**Par no projeto:** Passo 9 (CDC na prática) e Passo 4 (a dedup que faz o truque). Par na suíte: `test_deduplicacao_mantem_mais_recente`.

**Sumário**
1. Setup
2. O lote do dia (o "antes")
3. O update CDC (o mesmo gesto do script)
4. Reprocessar: o upsert absorvido
5. O contrafactual: e sem a dedup?
6. Ordem de chegada não importa (ordenação temporal importa)
7. Exercícios

> Convenção da pasta: roda de cima a baixo; dados fabricados; sem S3/LocalStack.

## 1. Setup

Spark + as funções reais do contrato + os helpers de fabricação dos testes:

In [ ]:
import sys

sys.path.insert(0, "../jobs")

from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[2]")
    .appName("lab-cdc")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

from contract import apply_contract, cast_types, deduplicate, split_valid_quarantine

COLS = ["event_id", "occurred_at", "customer_id", "merchant_id",
        "payment_method", "currency", "amount_cents", "status", "channel"]


def _row(**kwargs):
    base = {
        "event_id": "e1", "occurred_at": "2026-07-03T10:00:00", "customer_id": "cus_1",
        "merchant_id": "mer_1", "payment_method": "pix", "currency": "BRL",
        "amount_cents": "1000", "status": "approved", "channel": "app",
    }
    base.update(kwargs)
    return base


def raw_df(rows):
    return spark.createDataFrame(
        [tuple(str(r[c]) if r[c] is not None else None for c in COLS) for r in rows],
        schema=COLS,
    )


print("Spark", spark.version)

## 2. O lote do dia (o "antes")

Três transações aprovadas, de manhã. É o arquivo batch que já estaria na bronze:

In [ ]:
lote_batch = [
    _row(event_id="pg-001", occurred_at="2026-07-03T09:10:00", amount_cents="12000"),
    _row(event_id="pg-002", occurred_at="2026-07-03T09:30:00", amount_cents="8000",
         merchant_id="mer_2", payment_method="card"),
    _row(event_id="pg-003", occurred_at="2026-07-03T10:05:00", amount_cents="45000",
         merchant_id="mer_3"),
]
raw_df(lote_batch).select("event_id", "occurred_at", "status", "amount_cents").show()

## 3. O update CDC (o mesmo gesto do script)

À tarde, o cliente do `pg-002` pediu estorno. Um CDC real (DMS/Debezium) capturaria o UPDATE na origem e re-emitiria o registro: **mesmo `event_id`, conteúdo novo, timestamp maior**. É exatamente o que `generate_cdc_updates.py` fabrica — status vira `refunded`, `occurred_at` avança 1h:

In [ ]:
def gera_update(evento, horas=1):
    # espelho de ingest/generate_cdc_updates.py, sem o S3
    update = dict(evento)
    update["status"] = "refunded"
    occurred = datetime.fromisoformat(evento["occurred_at"])
    update["occurred_at"] = (occurred + timedelta(hours=horas)).isoformat()
    return update


update_cdc = gera_update(lote_batch[1])
print("original:", lote_batch[1]["occurred_at"], lote_batch[1]["status"])
print("update:  ", update_cdc["occurred_at"], update_cdc["status"])

## 4. Reprocessar: o upsert absorvido

No mundo real, o update cai na bronze ao lado do batch (`cdc-updates-*.jsonl.gz` no mesmo `dt`) e o silver do dia é **reprocessado inteiro**. Simulamos: batch + update entram juntos no contrato —

In [ ]:
reprocessado = apply_contract(deduplicate(cast_types(raw_df(lote_batch + [update_cdc]))))
validos, _quarentena = split_valid_quarantine(reprocessado)

validos.select("event_id", "occurred_at", "status").orderBy("event_id").show()

Cada `event_id` aparece **uma vez**, e o `pg-002` saiu `refunded` — a janela por `event_id` ordenada por `occurred_at` desc ficou com a versão mais recente. Nenhuma linha de código do pipeline mudou; o upsert foi absorvido **por construção**. Overwrite dinâmico da partição no `bronze_to_silver` garante a idempotência do reprocesso.

## 5. O contrafactual: e sem a dedup?

O que o negócio veria se o pipeline *não* deduplicasse — as duas versões do `pg-002` somadas:

In [ ]:
sem_dedup = apply_contract(cast_types(raw_df(lote_batch + [update_cdc])))
v_sem, _ = split_valid_quarantine(sem_dedup)

print("COM dedup (o pipeline real):")
(validos.groupBy("status").agg(F.count("*").alias("tx"),
                               F.sum("amount_cents_typed").alias("cents")).show())
print("SEM dedup (o bug):")
(v_sem.groupBy("status").agg(F.count("*").alias("tx"),
                             F.sum("amount_cents_typed").alias("cents")).show())

Sem a dedup, o `pg-002` conta **duas vezes** — uma como `approved`, outra como `refunded`: GMV inflado e estorno fantasma na mesma linha do relatório. É o duplo erro silencioso que a gold herdaria e o dashboard exibiria com toda confiança.

## 6. Ordem de chegada não importa (ordenação temporal importa)

Upsert por chave não depende de o update chegar *depois* no arquivo — depende do `occurred_at`. Invertendo a ordem de chegada (update antes do original), o resultado é o mesmo:

In [ ]:
invertido = apply_contract(deduplicate(cast_types(
    raw_df([update_cdc] + lote_batch)  # update chega "antes"
)))
v_inv, _ = split_valid_quarantine(invertido)
v_inv.select("event_id", "occurred_at", "status").orderBy("event_id").show()

Mesmo resultado: `pg-002` refunded, uma linha por id. É a diferença entre "o último que chegou ganha" (frágil: rede reordena, arquivo embaralha) e "o mais recente **no dado** ganha" (robusto: a verdade está no timestamp da origem).

## 7. Exercícios

1. **Update atrasado:** o estorno do `pg-002` só chega no dia seguinte, num arquivo com `dt=2026-07-04`. O reprocesso do dt 03 o enxerga? O que isso diz sobre janela de reprocessamento em pipelines com CDC?
2. **Empate de timestamp:** gere um update com o **mesmo** `occurred_at` do original. Quem ganha? A janela é determinística nesse caso? (Olhe o `desc_nulls_last` em `contract.py` e pense no que faltaria pra desempatar — um `sequence number` do CDC, por exemplo.)
3. **Delete (tombstone):** CDC também emite DELETE. Como você o representaria neste formato de evento — e o que mudaria no contrato pra linha sumir do silver sem sumir da bronze?
4. **Feche o ciclo:** o cenário da seção 6 (ordem de chegada invertida) não está na suíte. Escreva-o em `tests/test_contract.py` no padrão `_row` e rode `make test`.

In [ ]:
spark.stop()